# Model Tuning — Cross Validation, Grid Search CV & Randomized Search CV

This notebook is a **hands-on guide**, not just a script. Each section maps to a concept from the notes:

1. Baseline model (no tuning) — see the problem tuning solves
2. Cross Validation — a reliable way to score a model
3. Grid Search CV — try every hyperparameter combination
4. Grid Search CV on KNN — the worked example from the notes (`n_neighbors`, `weights`, `metric`)
5. Randomized Search CV — sample combinations instead of trying all of them
6. Grid vs Random — side-by-side comparison

Dataset: **Iris** (`sns.load_dataset('iris')`) — same dataset used throughout, 3 flower species, 4 numeric features.


## 0. Imports & Setup

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

sns.set_style("whitegrid")
RANDOM_STATE = 42

## 1. Load & Explore the Data

In [2]:
df = sns.load_dataset('iris')
print(df.shape)
df.head()

(150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [3]:
X = df.drop('species', axis=1)
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=RANDOM_STATE
)
print(f"Train size: {X_train.shape[0]}   Test size: {X_test.shape[0]}")

Train size: 100   Test size: 50


## 2. Baseline — Why We Can't Just "Eyeball" Hyperparameters

Train one SVM with default-ish settings and score it on a **single** train/test split.
The problem: this score depends on which rows happened to land in the test set —
a "lucky" split makes the model look better than it is, an "unlucky" one makes it look worse.


In [4]:
baseline_model = SVC(gamma='auto', C=1, kernel='rbf', random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
baseline_score = baseline_model.score(X_test, y_test)
print(f"Single-split accuracy: {baseline_score:.4f}")

Single-split accuracy: 1.0000


## 3. Cross Validation — A More Reliable Score

Instead of one split, **K-Fold CV** rotates which fold is the test set K times and averages
the K accuracy scores. This is the evaluation method Grid/Randomized Search use internally.


In [5]:
cv_scores = cross_val_score(
    SVC(gamma='auto', C=1, kernel='rbf'), X, y, cv=5
)
print("Per-fold accuracy:", np.round(cv_scores, 4))
print(f"Mean CV accuracy:  {cv_scores.mean():.4f}")
print(f"Std across folds:  {cv_scores.std():.4f}")

Per-fold accuracy: [0.9667 1.     0.9667 0.9667 1.    ]
Mean CV accuracy:  0.9800
Std across folds:  0.0163


## 4. Grid Search CV — Try Every Combination

`GridSearchCV` builds the **full cross-product** of every hyperparameter value you give it,
runs K-Fold CV on each combination, and keeps the best one.

Here we tune an SVM over `C` and `kernel`.


In [6]:
param_grid_svm = {
    'C': [1, 10, 20, 30],
    'kernel': ['rbf', 'linear'],
}

grid_svm = GridSearchCV(
    SVC(gamma='auto'), param_grid_svm, cv=5, return_train_score=False
)
grid_svm.fit(X, y)

results_svm = pd.DataFrame(grid_svm.cv_results_)
results_svm[['param_C', 'param_kernel', 'mean_test_score']] \
    .sort_values('mean_test_score', ascending=False) \
    .reset_index(drop=True)

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667
6,30,rbf,0.960000
7,30,linear,0.960000


In [7]:
print("Best params:", grid_svm.best_params_)
print(f"Best CV score: {grid_svm.best_score_:.4f}")

Best params: {'C': 1, 'kernel': 'rbf'}
Best CV score: 0.9800


## 5. Grid Search CV on KNN — the Notes' Worked Example

From the notes:
- `n_neighbors` = `[3, 5, 7, 9, 11, 13]`
- `weights` = `["uniform", "distance"]`
- `metric` = `["manhattan", "euclidean"]`

Total combinations = 6 × 2 × 2 = **24**, each evaluated with 5-fold CV.


In [8]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 13],
    'weights': ['uniform', 'distance'],
    'metric': ['manhattan', 'euclidean'],
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(), param_grid_knn, cv=5, return_train_score=False
)
grid_knn.fit(X, y)

print(f"Total combinations tested: {len(grid_knn.cv_results_['params'])}")
print("Best params:", grid_knn.best_params_)
print(f"Best CV score: {grid_knn.best_score_:.4f}")

results_knn = pd.DataFrame(grid_knn.cv_results_)
results_knn[['param_n_neighbors', 'param_weights', 'param_metric', 'mean_test_score']] \
    .sort_values('mean_test_score', ascending=False) \
    .head(10).reset_index(drop=True)

Total combinations tested: 24
Best params: {'metric': 'euclidean', 'n_neighbors': 11, 'weights': 'distance'}
Best CV score: 0.9867


,param_n_neighbors,param_weights,param_metric,mean_test_score
0,11,distance,euclidean,0.986667
1,11,uniform,euclidean,0.980000
2,13,distance,euclidean,0.980000
3,7,uniform,euclidean,0.980000
4,11,uniform,manhattan,0.980000
5,7,distance,euclidean,0.980000
6,9,uniform,manhattan,0.973333
7,7,uniform,manhattan,0.973333
8,9,distance,euclidean,0.973333
9,9,uniform,euclidean,0.973333


## 6. Randomized Search CV — Sample Instead of Exhaust

When the search space is huge (e.g. XGBoost with thousands of combinations), Grid Search
becomes too slow. `RandomizedSearchCV` samples a fixed number (`n_iter`) of random
combinations instead of testing all of them.

We re-run the same SVM grid, but only sample 4 out of the 8 possible combinations.


In [9]:
random_svm = RandomizedSearchCV(
    SVC(gamma='auto'), param_grid_svm,
    n_iter=4, cv=5, random_state=RANDOM_STATE, return_train_score=False
)
random_svm.fit(X, y)

print(f"Combinations tried: {len(random_svm.cv_results_['params'])} "
      f"(out of {4 * 2} possible)")
print("Best params:", random_svm.best_params_)
print(f"Best CV score: {random_svm.best_score_:.4f}")

Combinations tried: 4 (out of 8 possible)
Best params: {'kernel': 'linear', 'C': 1}
Best CV score: 0.9800


## 7. Grid vs Random — Side by Side

In [10]:
comparison = pd.DataFrame({
    'Method': ['Grid Search (SVM)', 'Randomized Search (SVM)'],
    'Combinations tried': [len(results_svm), len(random_svm.cv_results_['params'])],
    'Best score': [grid_svm.best_score_, random_svm.best_score_],
    'Best params': [grid_svm.best_params_, random_svm.best_params_],
})
comparison

,Method,Combinations tried,Best score,Best params
0,Grid Search (SVM),8,0.98,"{'C': 1, 'kernel': 'rbf'}"
1,Randomized Search (SVM),4,0.98,"{'kernel': 'linear', 'C': 1}"


## 8. Takeaways

- **Cross validation** gives a trustworthy score by rotating the test fold — never judge a model
  off a single train/test split.
- **Grid Search CV** is exhaustive and guarantees the best combination *within the grid you gave it* —
  but it gets expensive fast as you add more hyperparameters/values.
- **Randomized Search CV** samples a fixed budget of combinations — much cheaper, and usually finds
  a result close to Grid Search's, especially on large search spaces (e.g. XGBoost tuning).
- Rule of thumb: **small grid → Grid Search**, **large/high-dimensional grid → Randomized Search**.
